# L3: Controlled Generative UI
So far your agent has only replied in text. Now it can show React UI: flight cards, pie charts, anything you wire up. You build the components; the agent picks which to use and when.

## 📋 Learning Objectives
1. **Understand Generative UI** - What it is and where Controlled Generative UI sits on the spectrum
2. **Register frontend components** - Use `useComponent()` to expose React components to the agent
3. **Render structured tool output** - Let the agent choose and populate UI components in chat

## What you'll build

A chat interface that renders rich UI components — flight cards, pie charts — based on what the user asks for:

<div style="background: #f5f5f5; border: 1px solid #e0e0e0; border-radius: 20px; padding: 8px 16px; display: inline-block; font-size: 14px; color: #333; margin: 8px 0;">Show a flight card for Pacific Air from SFO to JFK departing at 08:30 for $249</div>

<img src="images/flight-card.png" alt="Flight Card" style="max-width: 600px; border: 1px solid #ddd; border-radius: 8px;" />

<div style="background: #f5f5f5; border: 1px solid #e0e0e0; border-radius: 20px; padding: 8px 16px; display: inline-block; font-size: 14px; color: #333; margin: 8px 0;">Please show me the distribution of our revenue by category in a pie chart</div>

<img src="images/pie-chart.png" alt="Pie Chart" style="max-width: 600px; border: 1px solid #ddd; border-radius: 8px;" />

## What is Controlled Generative UI?
Generative UI is a pattern where agents respond with fully interactive interfaces, not just text. **Controlled Generative UI** is the most constrained variant: the agent can only render components you explicitly register.

### How it works
Each registered component is exposed as a tool with:
- a stable name
- a typed input schema
- a mapped React component

The agent doesn't generate arbitrary UI. It passes structured data into components you've built, all defined on the frontend and registered at runtime.

### Pros and cons
**Pros**
- Easy to implement: register a component and you're done.
- High visual polish, since every rendered surface is one you authored.
- Strong safety: the model can only call registered tools with validated arguments.
- Good fit for high-traffic or mission-critical UX where stability matters.

**Cons**
- Frontend effort grows with every new capability: each pattern needs its own component.
- Less expressive freedom than declarative or open-ended generative UI.

### The `useComponent()` hook
[useComponent](https://docs.copilotkit.ai/generative-ui/your-components/display-only) registers a React component as a tool the agent can call inside `<CopilotChat />`. You define what's available; the agent picks when to use it.

```tsx
useComponent({
  name: "component_name",
  description: "description for the agent to know about this component",
  parameters: z.object({ ... }),
  render: MyComponent,
});
```

- `name` (required `string`): tool name exposed to the model.
- `description` (optional `string`): tells the model when to call this tool.
- `parameters` (optional Zod schema): structured props passed in as arguments.
- `render` (required): a React component (rendered as `<Component {...args} />`), or a function receiving `{ args, status }` for custom rendering (loading states, wrappers, conditional display).

In [ ]:
# clear your state if running notebook more than once
from helper import reset_lesson
reset_lesson(3)

## Setup Dependencies

In [ ]:
# install dependencies
# !pip install -r requirements.txt

In [ ]:
from helper import load_api_keys
load_api_keys()

## Start Agent

In [ ]:
# Start the same agent backend from L2:
from backend.server import start_backend
start_backend(port=8003)

## Start Frontend

In [ ]:
# start front end
from helper import start_frontend
start_frontend(port=3003)

In [ ]:
# preview the app
from helper import display_app
display_app(port=3003)

## Register components

The example below registers three components with `useComponent()`: a simple `showName` card, a pie chart, and a flight card.

In [ ]:
%%writefile frontend/src/App.tsx
import { z } from "zod"
import { CopilotChat } from "@copilotkit/react-core/v2";
import { useComponent } from "@copilotkit/react-core/v2";

import { FlightCard, FlightCardProps } from "@/components/flight-card";
import { PieChart, PieChartProps } from "@/components/pie-chart";

import { useExampleSuggestions } from "@/hooks/use-example-suggestions";

export default function App() {

  // 🪁 Register a component that shows the name of the user
  useComponent({
    name: "showMyName",
    description: "Show the user's name in a card",
    parameters: z.object({ name: z.string() }),
    render: ({ name }) => <div className="bg-blue-500 p-4">Hi, {name}!</div>,
  });

  // 🪁 Resgister a pieChart component to show structured data
  useComponent({
    name: "pieChart",
    description: "Controlled Generative UI that displays data as a pie chart.",
    parameters: PieChartProps,
    render: PieChart,
  });

  // 🪁 Resgister a flightCard component to show flight data
  useComponent({
    name: "flightCard",
    description: "Controlled Generative UI that displays a single flight summary card.",
    parameters: FlightCardProps,
    render: FlightCard,
  });

  // 🪁 Add suggestions to our CopilotChat, will display through buttons
  useExampleSuggestions();

  return <CopilotChat />;

};

## Display the app

Your agent now has three components it can render. Try prompting it to use each one.
<div style="background: #f5f5f5; border: 1px solid #e0e0e0; border-radius: 20px; padding: 8px 16px; display: inline-block; font-size: 14px; color: #333; margin: 8px 0;">Show my name</div>
<div style="background: #f5f5f5; border: 1px solid #e0e0e0; border-radius: 20px; padding: 8px 16px; display: inline-block; font-size: 14px; color: #333; margin: 8px 0;">Pie chart</div>
<div style="background: #f5f5f5; border: 1px solid #e0e0e0; border-radius: 20px; padding: 8px 16px; display: inline-block; font-size: 14px; color: #333; margin: 8px 0;">Flight card</div>


In [ ]:
from helper import display_app
display_app(port=3003)

### Recap

Any React component can become a generative UI tool — just register it with `useComponent()`. Here's the full `FlightCard` for reference:
```tsx
import { z } from "zod";

export const FlightCardProps = z.object({
  title: z.string().describe("Flight card title"),
  airline: z.string().describe("Airline name"),
  origin: z.string().describe("Departure airport/city"),
  destination: z.string().describe("Arrival airport/city"),
  departure_time: z.string().describe("Departure time"),
  price: z.string().describe("Price display"),
});

type FlightCardProps = z.infer<typeof FlightCardProps>;

export function FlightCard({
  title,
  airline,
  origin,
  destination,
  departure_time,
  price,
}: FlightCardProps) {
  return (
    <div className="rounded-lg border bg-white p-3 space-y-2">
      <div className="font-semibold">{title}</div>
      <div className="rounded border p-2 text-sm">
        <div className="font-medium">{airline}</div>
        <div>
          {origin} → {destination}
        </div>
        <div>Departs: {departure_time}</div>
        <div className="font-semibold">{price}</div>
      </div>
    </div>
  );
}
```

**Why Zod?**

[Zod](https://zod.dev) is a TypeScript-first schema and validation library — similar to [Pydantic](https://docs.pydantic.dev) in Python. It lets you define the component props once and use the same schema for both the TypeScript type definition and the `parameters` passed to `useComponent()`.

## What you learned

- Controlled generative UI limits rendering to explicitly registered components.
- `useComponent()` defines a typed contract for the tool arguments that are wired up to the React component props.
- The backend and frontend split responsibilities cleanly: the agent chooses tools and data, while the UI layer owns rendering.
- This produces a safer, more predictable agent experience than open-ended UI generation, while sacrificing some flexibility.


<div style="background-color:#fff1d7; padding:15px; border-left: 4px solid #f0b429; border-radius: 6px;">
<b>Try it yourself:</b> Register a new component (e.g., a table or a progress bar) with <code>useComponent</code> and prompt the agent to use it. You'll need to define a Zod schema for the props and create a React component that renders them.
</div>

## Dig Deeper

Want to explore controlled generative UI further? Here are some resources:

- **[Headless CopilotKit](https://docs.copilotkit.ai/langgraph/custom-look-and-feel/headless-ui)** — Run CopilotKit without the built-in chat UI for custom interfaces.
- **[Frontend Tools Reference](https://docs.copilotkit.ai/langgraph/generative-ui/frontend-tools)** — Full `useComponent()` API reference and advanced patterns.
- **[AG-UI Protocol](https://docs.ag-ui.com)** — The protocol that connects CopilotKit to any agent backend.

## Next step

In **Lesson 4**, you'll move to **declarative generative UI** — the middle of the spectrum. Instead of registering individual components, you'll define a catalog of building blocks and let the agent compose them into layouts using the A2UI spec.